In [1]:
from compas_dem.material import Stone
from compas_dem.models import BlockModel
from compas_dem.problem import Problem
from compas_dem.templates import ArchTemplate
from compas_dem.viewer import DEMViewer  # noqa: E402
from compas_model.materials import Concrete


In [2]:

# Block Model using compas_dem's Arch Template
template: ArchTemplate = ArchTemplate(rise=4.393, span=21.213, thickness=0.5, depth=3.0, n=100)
model: BlockModel = BlockModel.from_template(template)

# Compute contacts to populate the graph with contact properties for each interface.
model.compute_contacts()



# Instantiate materials and add to the model
conc: Concrete = Concrete.from_strength_class("C30")
conc.density = 2000.0
model.add_material(conc)

limestone: Stone = Stone.from_predefined_material("LimeStone")
limestone.density = 2000.0
model.add_material(limestone)

# Print some properties to check everything is working
print(f"Limestone properties: Ecm={limestone.Ecm}, poisson={limestone.poisson}, density={limestone.density}")

# Assign materials to blocks based on centroid z-coordinate
for element in model.blocks():
    if element.modelgeometry.centroid()[2] > 0.5:
        model.assign_material(conc, element)
    else:
        model.assign_material(limestone, element)


Limestone properties: Ecm=20000, poisson=0.2, density=2000.0


In [3]:

# =============================================================================
# Create a problem instance
# =============================================================================


problem = Problem(model)

# Create Contact model and Joint model instances
# -----------------------------------------------

# mohr_columb: MohrCoulomb = MohrCoulomb(phi=30, c=0)  # Mohr Columb is a contact model, thus it inherits from ContactModel, and can be assigned to the interfaces
# Other contact models can be simply added by inheriting from ContactModel and implementing the required methods/ parameters

# print(f"Contact model properties: phi={mohr_columb.phi}, c={mohr_columb.c}, mu={mohr_columb.mu}")

# joint_a: JointModel = JointModel(kn=10e10, kt=10e7)

# problem.add_contact_model("MohrCoulomb", phi=30, c=0)
# problem.add_joint_model("JointA", kn=10e10, kt=10e7)

# problem.add_contact(contact_model=mohr_columb, joint_model=joint_a)

# Create Contact Properties using the Contact and Joint models
# -------------------------------------------------------------


# contact_type_1 = ContactProperties(contact_model=mohr_columb, joint_model=joint_a)

problem.add_contact_model("MohrCoulomb", phi=30, c=0)
# problem.add_contact(contact_type_1)

# from compas_dem.problem import BoundaryConditions  # noqa: E402

# bc = BoundaryConditions()
# bc.add_point_load(block_index=70, force=[0, 0, -151500])
# bc.add_fixed(block_index=0)
# bc.add_fixed(block_index=99)

# problem.apply_bc(bc)$
# problem.add_displacement(block_index=50, dz=-0.01)
problem.add_point_load(block_index=70, force=[0, 0, -151500])
problem.add_supports_from_model()
# keep BC class, but populate into problem directly, then add the data to BC dict inside problem.

# Solve the problem using the LMGC90 solver
# -----------------------------------------

solution = problem.solve(solver="LMGC90", duration=1.0, n_steps=100, urf_threshold=0.001 )

# =============================================================================
# Visualize the model in the DEM Native viewer
# =============================================================================

# force_time = solution.force_time
# mean_forces = [np.mean(f) if len(f) > 0 else 0.0 for f in force_time]

# plt.figure(figsize=(10, 5))
# plt.plot(mean_forces)
# plt.xlabel("Time step")
# plt.ylabel("Mean contact force magnitude")
# plt.title("Contact force magnitudes over time")
# plt.tight_layout()
# plt.show()


# viewer = DEMViewer(model)
# viewer.add_solution("LMGC90", solution, scale_force=10e-7, scale_normal=0.0000001)  # Passing the scale KWARGS specific to LMGC90
# viewer.show()


[[-1.14052915e-18  1.00000000e+00  0.00000000e+00]
 [-7.01490983e-01 -1.11022302e-16 -7.12678329e-01]
 [-7.12678329e-01 -8.12830412e-19  7.01490983e-01]]
Starting LMGC90 solver analysis...
Completed step 0/100...  UFR = 9.84e-01
Completed step 10/100...  UFR = 5.92e-01
Completed step 20/100...  UFR = 4.51e-01
Completed step 30/100...  UFR = 8.13e-01
Completed step 40/100...  UFR = 1.00e+00
Completed step 50/100...  UFR = 1.00e+00
Completed step 60/100...  UFR = 1.00e+00
Completed step 70/100...  UFR = 1.00e+00
Diverged at step 70 (UFR = 1.00e+00 >= 1.0). Stopping.
LMGC90 solver run complete.


In [4]:
for edge in model.graph.edges():
    print(model.graph.edge_attribute(edge, "force"))

None
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.

In [5]:
viewer = DEMViewer(model)
viewer.add_solution(solution=solution, scale_force=10e-7)  # Passing the scale KWARGS specific to LMGC90
viewer.show()

solution.finalize()


ZeroDivisionError: float division by zero